<a href="https://colab.research.google.com/github/SRIKRISHNA1407627/BIS_LAB_1BM24CS151/blob/main/WEEK_03_ANTCOLONY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import random

class AgriculturalDroneACO:
    def __init__(self, num_regions, distance_matrix, energy_rate=1.2, max_battery=100.0,
                 num_ants=20, generations=100, alpha=1.0, beta=2.0, evaporation_rate=0.5, Q=100):
        """
        Initializes the Ant Colony Optimization algorithm for battery-constrained drone routing.

        :param num_regions: Total number of regions including the charging station (Index 0).
        :param distance_matrix: 2D array representing distances between regions.
        :param energy_rate: Energy consumed per unit of distance.
        :param max_battery: Maximum battery capacity of the drone.
        """
        self.num_regions = num_regions
        self.distance_matrix = distance_matrix
        self.energy_rate = energy_rate
        self.max_battery = max_battery

        # ACO Parameters
        self.num_ants = num_ants
        self.generations = generations
        self.alpha = alpha                # Pheromone exponential weight (importance)
        self.beta = beta                  # Heuristic exponential weight (nearness importance)
        self.evaporation_rate = evaporation_rate
        self.Q = Q                        # Pheromone deposit scaling constant

        # Initialize pheromone matrix with small uniform values
        self.pheromone_matrix = np.ones((num_regions, num_regions)) * 0.1

        # Heuristic matrix (Inverse of distance; offset small values to avoid division by zero)
        self.heuristic_matrix = 1.0 / (distance_matrix + np.eye(num_regions) * 1e-6)

    def _select_next_region(self, current_region, unvisited, current_battery):
        """Selects the next region based on pheromone trails, proximity, and battery constraints."""
        safe_unvisited = []

        # Safety Check: Confirm the drone can visit the candidate field AND return home safely
        for next_reg in unvisited:
            energy_to_next = self.distance_matrix[current_region][next_reg] * self.energy_rate
            energy_to_home = self.distance_matrix[next_reg][0] * self.energy_rate

            if current_battery >= (energy_to_next + energy_to_home):
                safe_unvisited.append(next_reg)

        # If no unvisited region is within safe reach, the ant is forced back to charging station (0)
        if not safe_unvisited:
            return 0

        # Calculate transition probabilities for all safe, unvisited paths
        probabilities = []
        for next_reg in safe_unvisited:
            pheromone = self.pheromone_matrix[current_region][next_reg] ** self.alpha
            heuristic = self.heuristic_matrix[current_region][next_reg] ** self.beta
            probabilities.append(pheromone * heuristic)

        sum_probs = sum(probabilities)
        if sum_probs == 0:
            return random.choice(safe_unvisited)

        probabilities = [p / sum_probs for p in probabilities]

        # Stochastic selection based on calculated weights
        return random.choices(safe_unvisited, weights=probabilities, k=1)[0]

    def _construct_ant_path(self):
        """Constructs a complete valid routing sequence for a single ant agent."""
        path = [0]  # Every ant starts at the charging station (0)
        unvisited = set(range(1, self.num_regions))
        current_battery = self.max_battery
        total_distance = 0
        total_energy = 0

        while unvisited:
            current_region = path[-1]
            next_region = self._select_next_region(current_region, unvisited, current_battery)

            distance = self.distance_matrix[current_region][next_region]
            energy_spent = distance * self.energy_rate

            total_distance += distance
            total_energy += energy_spent

            if next_region == 0:
                # The drone visits the charging station and resets battery capacity
                current_battery = self.max_battery
            else:
                # The drone visits a region field
                current_battery -= energy_spent
                unvisited.remove(next_region)

            path.append(next_region)

        # Guarantee that the path terminates neatly back at the base station
        if path[-1] != 0:
            distance_to_home = self.distance_matrix[path[-1]][0]
            total_distance += distance_to_home
            total_energy += distance_to_home * self.energy_rate
            path.append(0)

        return path, total_distance, total_energy

    def solve(self):
        """Executes the complete generation loop for the Ant Colony Optimization algorithm."""
        best_path = None
        best_distance = float('inf')
        best_energy = float('inf')

        for gen in range(self.generations):
            all_paths = []

            for ant in range(self.num_ants):
                path, distance, energy = self._construct_ant_path()
                all_paths.append((path, distance, energy))

                # Check for global optimization metrics
                if distance < best_distance:
                    best_distance = distance
                    best_energy = energy
                    best_path = path

            # Global Evaporation Step
            self.pheromone_matrix *= (1 - self.evaporation_rate)

            # Pheromone Update Rule (Reinforcing edges proportional to path effectiveness)
            for path, distance, _ in all_paths:
                for i in range(len(path) - 1):
                    u, v = path[i], path[i+1]
                    self.pheromone_matrix[u][v] += self.Q / distance
                    self.pheromone_matrix[v][u] += self.Q / distance  # Symmetric route link updating

        return best_path, best_distance, best_energy


# --- Execution Pipeline & Demo Setup ---
if __name__ == "__main__":
    np.random.seed(42)
    random.seed(42)

    # 6 Regions defined (Index 0 = Hub/Station, Indexes 1 to 5 = Farm Locations)
    num_fields = 6

    # Create spatial coordinates distributed randomly within a farm plot mapping
    coordinates = np.random.rand(num_fields, 2) * 100

    # Precompute structural distance matrix array
    dist_matrix = np.zeros((num_fields, num_fields))
    for i in range(num_fields):
        for j in range(num_fields):
            dist_matrix[i][j] = np.linalg.norm(coordinates[i] - coordinates[j])

    print("--- Calculated Field Distance Matrix ---")
    print(np.round(dist_matrix, 2))

    # Instantiate and evaluate optimization sequence
    drone_optimizer = AgriculturalDroneACO(
        num_regions=num_fields,
        distance_matrix=dist_matrix,
        energy_rate=1.4,       # Consumption coefficient multiplier per meter
        max_battery=100.0,     # Total system starting percentage
        num_ants=20,
        generations=60
    )

    optimal_sequence, optimized_distance, optimized_energy = drone_optimizer.solve()

    print("\n--- ACO Engine Global Best Solution ---")
    print(
        f"**Optimal Visiting Sequence:** {' -> '.join(map(str, optimal_sequence))}\n"
        f"**Total Flight Distance:** {optimized_distance:.2f} units\n"
        f"**Total Energy Consumption Run:** {optimized_energy:.2f}% Battery Units Consumed"
    )


--- Calculated Field Distance Matrix ---
[[ 0.   50.17 82.42 32.76 33.2  35.45]
 [50.17  0.   72.64 72.51 17.06 80.25]
 [82.42 72.64  0.   71.69 70.92 82.51]
 [32.76 72.51 71.69  0.   56.56 11.03]
 [33.2  17.06 70.92 56.56  0.   63.68]
 [35.45 80.25 82.51 11.03 63.68  0.  ]]
